# Качество треков

Ноутбук готовит данные для дашборда качества каталога. Одна строка
витрины соответствует одному `item_id`.

## Методика

Основные признаки качества - Listen+, повторы, лайки и дизлайки на
1000 прослушиваний. Для рейтингов используется только надёжная выборка:
не менее 100 прослушиваний на трек. Это защищает от случайных 0% и 100%
на единичных наблюдениях.

In [28]:
from pathlib import Path
import sys

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

STAGE_DB = PROJECT_ROOT / "data" / "interim" / "yambda_stage.duckdb"
MARTS = PROJECT_ROOT / "data" / "processed"
assert STAGE_DB.exists(), "Сначала выполните ноутбук 01_source_quality.ipynb"
MARTS.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
con = duckdb.connect()

## Сборка витрины

In [29]:
import importlib

import src.mart_settings as mart_settings_module
import src.track_mart as track_mart_module

importlib.reload(mart_settings_module)
track_mart_module = importlib.reload(track_mart_module)

build_track_mart = track_mart_module.build_track_mart
validate_all_marts = track_mart_module.validate_all_marts

TRACK_MART = MARTS / "mart_track.parquet"
PRODUCT_MART = MARTS / "mart_product_day.parquet"
USER_MART = MARTS / "mart_user.parquet"
USER_DAY_MART = MARTS / "mart_user_day.parquet"
RETENTION_MART = MARTS / "mart_retention_cohort.parquet"
FUNNEL_MART = MARTS / "mart_recommendation_funnel.parquet"

build_result = build_track_mart(STAGE_DB, TRACK_MART)
track_columns = set(
    con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{TRACK_MART.as_posix()}')"
    ).df()["column_name"]
)
required_columns = {
    "is_reliable_sample",
    "recommendation_listen_plus",
    "organic_listen_plus",
}
missing_columns = required_columns - track_columns
assert not missing_columns, f"В витрине не хватает столбцов: {sorted(missing_columns)}"
build_result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'rows': 934057}

## Размер и качество каталога

In [30]:
track_summary = con.execute(f'''
SELECT
    count(*) AS tracks,
    count(*) FILTER (WHERE listens > 0) AS listened_tracks,
    count(*) FILTER (WHERE is_reliable_sample) AS reliable_tracks,
    sum(listens) AS listens,
    sum(listen_plus) * 100.0 / sum(listens) AS listen_plus_pct,
    sum(replays) * 100.0 / sum(listens) AS replay_pct,
    sum(likes) * 1000.0 / sum(listens) AS likes_per_1000,
    sum(dislikes) * 1000.0 / sum(listens) AS dislikes_per_1000,
    median(listeners) FILTER (WHERE listens > 0) AS median_listeners
FROM read_parquet('{TRACK_MART.as_posix()}')
''').df().round(2)
track_summary.rename(columns={
    "tracks": "все треки", "listened_tracks": "прослушанные треки",
    "reliable_tracks": "треки с 100+ прослушиваниями",
    "listens": "прослушивания", "listen_plus_pct": "Listen+, %",
    "replay_pct": "повторы, %", "likes_per_1000": "лайки на 1000",
    "dislikes_per_1000": "дизлайки на 1000",
    "median_listeners": "медиана слушателей",
})

,все треки,прослушанные треки,треки с 100+ прослушиваниями,прослушивания,"Listen+, %","повторы, %",лайки на 1000,дизлайки на 1000,медиана слушателей
0,934057,877168,64105,"46,467,212.00",63.21,0.47,18.97,2.32,2.00


## Качество рекомендаций и органики

In [31]:
track_source_quality = con.execute(f'''
SELECT
    sum(recommendation_listen_plus) * 100.0
        / sum(recommendation_listens) AS recommendation_listen_plus_pct,
    sum(organic_listen_plus) * 100.0
        / sum(organic_listens) AS organic_listen_plus_pct,
    sum(recommendation_listens) * 100.0 / sum(listens)
        AS recommendation_share_pct
FROM read_parquet('{TRACK_MART.as_posix()}')
''').df().round(2)
track_source_quality.rename(columns={
    "recommendation_listen_plus_pct": "Listen+ рекомендаций, %",
    "organic_listen_plus_pct": "Listen+ органики, %",
    "recommendation_share_pct": "доля рекомендаций, %",
})

,"Listen+ рекомендаций, %","Listen+ органики, %","доля рекомендаций, %"
0,68.06,58.67,48.34


## Сегменты качества

In [32]:
quality_segments = con.execute(f'''
WITH reliable AS (
    SELECT * FROM read_parquet('{TRACK_MART.as_posix()}')
    WHERE is_reliable_sample
),
limits AS (
    SELECT
        quantile_cont(listen_plus_rate, 0.25) AS listen_plus_q25,
        quantile_cont(listen_plus_rate, 0.75) AS listen_plus_q75,
        quantile_cont(dislikes_per_1000_listens, 0.75) AS dislikes_q75
    FROM reliable
),
segmented AS (
    SELECT
        r.*,
        CASE
            WHEN listen_plus_rate >= listen_plus_q75
                 AND dislikes_per_1000_listens < dislikes_q75
                THEN 'высокое качество'
            WHEN listen_plus_rate <= listen_plus_q25
                 OR dislikes_per_1000_listens >= dislikes_q75
                THEN 'требует внимания'
            ELSE 'стабильные'
        END AS quality_segment
    FROM reliable r CROSS JOIN limits
)
SELECT
    quality_segment,
    count(*) AS tracks,
    sum(listens) AS listens,
    avg(listen_plus_rate) * 100 AS avg_listen_plus_pct,
    avg(dislikes_per_1000_listens) AS avg_dislikes_per_1000
FROM segmented
GROUP BY quality_segment
ORDER BY tracks DESC
''').df().round(2)
quality_segments.rename(columns={
    "quality_segment": "сегмент", "tracks": "треки",
    "listens": "прослушивания", "avg_listen_plus_pct": "средний Listen+, %",
    "avg_dislikes_per_1000": "средние дизлайки на 1000",
})

,сегмент,треки,прослушивания,"средний Listen+, %",средние дизлайки на 1000
0,требует внимания,27024,"12,357,517.00",56.45,4.54
1,стабильные,24152,"20,492,865.00",64.27,0.77
2,высокое качество,12929,"6,141,014.00",80.86,0.46


## Популярные треки

In [33]:
popular_tracks = con.execute(f'''
SELECT
    item_id,
    listens,
    listeners,
    round(listen_plus_rate * 100, 2) AS listen_plus_pct,
    round(recommendation_share * 100, 2) AS recommendation_pct,
    round(likes_per_1000_listens, 2) AS likes_per_1000,
    round(dislikes_per_1000_listens, 2) AS dislikes_per_1000
FROM read_parquet('{TRACK_MART.as_posix()}')
WHERE is_reliable_sample
ORDER BY listens DESC
LIMIT 10
''').df()
popular_tracks

,item_id,listens,listeners,listen_plus_pct,recommendation_pct,likes_per_1000,dislikes_per_1000
0,5862961,41984,4699,67.19,23.27,18.75,1.95
1,6901374,40776,5350,66.09,31.55,20.16,1.99
2,3542184,39947,5046,65.40,27.45,20.80,4.26
3,9378983,38819,4836,64.87,25.68,20.69,2.32
4,5635052,38804,4652,65.65,30.59,15.80,2.35
5,5463340,38166,4417,61.02,23.71,19.60,1.76
6,8213481,37579,4356,63.26,24.80,14.16,3.65
7,906358,35870,3954,64.40,24.44,19.07,2.73
8,3033749,29977,4340,57.58,25.51,12.74,2.64
9,2859641,29602,3746,63.90,24.90,14.32,2.63


## Треки, требующие внимания

In [34]:
problem_tracks = con.execute(f'''
SELECT
    item_id,
    listens,
    listeners,
    round(listen_plus_rate * 100, 2) AS listen_plus_pct,
    round(dislikes_per_1000_listens, 2) AS dislikes_per_1000,
    round(recommendation_share * 100, 2) AS recommendation_pct
FROM read_parquet('{TRACK_MART.as_posix()}')
WHERE is_reliable_sample
ORDER BY listen_plus_rate ASC, dislikes_per_1000_listens DESC
LIMIT 10
''').df()
problem_tracks

,item_id,listens,listeners,listen_plus_pct,dislikes_per_1000,recommendation_pct
0,7626331,2908,1,0.00,0.00,0.00
1,8706789,320,4,0.94,0.00,0.31
2,8395381,557,3,1.08,0.00,0.00
3,3398319,264,5,1.52,0.00,1.52
4,5892681,197,149,2.54,0.00,10.15
5,7599851,154,81,3.25,0.00,0.00
6,7115536,169,2,3.55,0.00,0.00
7,7569222,2075,25,3.71,0.00,0.29
8,8725946,161,21,3.73,0.00,91.93
9,6735307,190,123,4.74,15.79,0.00


## Общая сверка витрин

In [35]:
checks = validate_all_marts(
    STAGE_DB,
    PRODUCT_MART,
    USER_MART,
    USER_DAY_MART,
    RETENTION_MART,
    TRACK_MART,
    FUNNEL_MART,
)
assert all(checks.values())
checks

{'product_events_match_source': True,
 'product_listens_match_source': True,
 'user_key_is_unique': True,
 'user_day_key_is_unique': True,
 'user_day_listens_match_source': True,
 'retention_key_is_unique': True,
 'retention_rates_are_valid': True,
 'track_key_is_unique': True,
 'funnel_is_valid': True}

## Что готово для дашборда

- карточки размера и качества каталога;
- сравнение Listen+ рекомендаций и органики;
- сегменты высокого, стабильного и проблемного качества;
- рейтинги популярных и проблемных треков;
- непрерывные показатели для scatter-графика «прослушивания - Listen+».

В датасете нет названий, исполнителей и жанров, поэтому детализация
возможна только по `item_id` и длительности трека.

In [37]:
con.close()
print("Соединение закрыто")

Соединение закрыто
